In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split,GridSearchCV ## 資料切分函式庫
from sklearn.pipeline import Pipeline ## 建立機器學習流程的函式庫
from sklearn.preprocessing import StandardScaler ## 資料預處理函式庫
from sklearn.linear_model import LinearRegression, Ridge  ## 線性回歸和 Ridge 回歸模型
from sklearn.ensemble import RandomForestClassifier ## 隨機森林分類器
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score ## 評估指標函式庫

In [4]:
# ==============================================================================
# 專題名稱: 慢性病風險預測基準模型 (Centralized Baseline)
# 任務類型: 二元分類 (Binary Classification) - 預測糖尿病風險
# 演算法: 隨機森林 (Random Forest Classifier)
# 數據集: NHANES 2021-2023 (original_nhanes_cleaned_21_23.csv)
# ==============================================================================

# --- [1] 數據載入與清理 ---
# 載入 NHANES 21-23 週期數據
test_data = pd.read_csv('./data_set/original_nhanes_cleaned_21_23.csv')

# 移除無效標籤 (9.0 = 拒絕回答/未知)，並重設索引以保持數據結構連續性
test_data = test_data[test_data['DIQ010'] != 9.0].reset_index(drop=True)

# --- [2] 標籤工程 (Label Engineering) ---
# 將多分類問題簡化為「二元分類」，以處理醫療數據常見的類別不平衡問題
# 原始標籤定義: 1.0(有糖尿病), 2.0(無糖尿病), 3.0(邊緣性糖尿病)
# 轉換邏輯: 
#   - 健康組 (Class 0): 原本的 2.0
#   - 風險組 (Class 1): 原本的 1.0 與 3.0 合併
test_data['DIQ010'] = test_data['DIQ010'].replace({3.0: 1.0, 2.0: 0.0})

# --- [3] 特徵與標籤分離 ---
# X: 移除患者 ID (SEQN) 與 目標標籤 (DIQ010)，僅保留生理特徵
X = test_data.drop(columns=['SEQN', 'DIQ010'])
# Y: 目標變數 (0 或 1)
Y = test_data['DIQ010']

# --- [4] 數據集切分 ---
# 將資料隨機切分為 80% 訓練集與 20% 測試集
# random_state=42 確保每次切分的結果一致，利於後續調整超參數時的客觀比較
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
   
# --- [5] 特徵縮放 (Feature Scaling) ---
# 確保不同單位的生理指標 (如年齡、血糖、膽固醇) 對模型有平等的權重
scaler = StandardScaler()
# 嚴格遵守防資料外洩原則：fit 僅能計算訓練集的統計分佈
X_train_scaled = scaler.fit_transform(X_train) 
# 測試集僅執行 transform，模擬真實世界中遇到未知數據的情況
X_test_scaled = scaler.transform(X_test)

# --- [6] 模型建立與訓練 ---
# 初始化隨機森林分類器
model = RandomForestClassifier(random_state=42)
# 模型基於標準化後的特徵與二元標籤進行權重學習
model.fit(X_train_scaled, y_train)

# --- [7] 模型預測 ---
# 模型對未看過的測試集進行推理，輸出預測類別 (0 或 1)
predictions = model.predict(X_test_scaled)

# --- [8] 效能評估指標 (Metrics Evaluation) ---
# [8.1] 整體準確率 (Accuracy)
score = accuracy_score(y_test, predictions)
print(f"模型整體準確率: {score * 100:.2f}%\n")

# [8.2] 混淆矩陣 (Confusion Matrix)
# 用於觀察 True Positive, False Positive, True Negative, False Negative 的具體分佈
cm = confusion_matrix(y_test, predictions)
print("--- 混淆矩陣 (Confusion Matrix) ---")
print(cm)

# [8.3] 分類報告 (Classification Report)
# 深入評估各類別的 Precision (精準率), Recall (召回率) 與 F1-score
# 在醫療預測中，特別需關注 Risk(1) 類別的 Recall 表現
target_names = ['Health (0)', 'Risk (1)'] 
print("\n--- 分類評估報告 ---")
print(classification_report(y_test, predictions, target_names=target_names, zero_division=0))

# --- [9] 結果視覺化與存檔 ---
# 繪製混淆矩陣熱力圖 (Heatmap) 以供專題報告使用
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, 
            yticklabels=target_names)
plt.title('Binary Classification Confusion Matrix (Random Forest)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

# 檢查儲存目錄是否存在，若無則自動建立
if not os.path.exists("./SK_output"): 
    os.makedirs("./SK_output")

# 將圖片存檔並清除畫布緩存
plt.savefig("./SK_output/confusion_matrix.png")
plt.clf()

模型整體準確率: 92.39%

--- 混淆矩陣 (Confusion Matrix) ---
[[549  12]
 [ 39  70]]

--- 分類評估報告 ---
              precision    recall  f1-score   support

  Health (0)       0.93      0.98      0.96       561
    Risk (1)       0.85      0.64      0.73       109

    accuracy                           0.92       670
   macro avg       0.89      0.81      0.84       670
weighted avg       0.92      0.92      0.92       670



<Figure size 800x600 with 0 Axes>